In [1]:
print("ok")

ok


In [2]:
%pwd

'c:\\Users\\ROSY PAUL\\Desktop\\RagProject\\MedAssist-RAG\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\ROSY PAUL\\Desktop\\RagProject\\MedAssist-RAG'

In [8]:
from langchain_community.document_loaders import PyPDFLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
# Extracting text from pdf file
def load_pdf_files(data):
    loader=DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents=loader.load()
    return documents

In [10]:
extracted_data=load_pdf_files("data")

In [11]:
len(extracted_data)

637

In [13]:
extracted_data[0]

Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data\\Medical_book.pdf', 'total_pages': 637, 'page': 0, 'page_label': '1'}, page_content='')

In [21]:
from langchain_core.documents import Document
from typing import List
def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document] =[]

    for doc in docs:
        src=doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [22]:
minimal_docs=filter_to_minimal_docs(extracted_data)

In [24]:
minimal_docs[0]

Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='')

In [25]:
# split the documents into smaller chunks
def text_splitter(minimal_docs):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
        length_function=len
    )
    texts_chunk=text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [26]:
text_chunks=text_splitter(minimal_docs)
print(f"Number of chunks: {len(text_chunks)}")

Number of chunks: 5860


In [27]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"}
    )
    return embeddings

In [29]:
embeddings=download_embeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 605.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [30]:
vector=embeddings.embed_query("Hello world")

In [31]:
vector

[-0.034477293491363525,
 0.031023185700178146,
 0.006734919268637896,
 0.026108955964446068,
 -0.03936200588941574,
 -0.16030244529247284,
 0.06692398339509964,
 -0.00644145580008626,
 -0.047450482845306396,
 0.014758873730897903,
 0.07087531685829163,
 0.05552761256694794,
 0.019193336367607117,
 -0.026251327246427536,
 -0.010109526105225086,
 -0.026940451934933662,
 0.02230745740234852,
 -0.02222668007016182,
 -0.14969263970851898,
 -0.017492998391389847,
 0.007676251698285341,
 0.05435226485133171,
 0.003254401497542858,
 0.031725890934467316,
 -0.08462139964103699,
 -0.029405971989035606,
 0.051595598459243774,
 0.04812406003475189,
 -0.003314854810014367,
 -0.05827920511364937,
 0.04196922481060028,
 0.022210687398910522,
 0.1281888335943222,
 -0.02233893983066082,
 -0.011656275019049644,
 0.06292839348316193,
 -0.032876357436180115,
 -0.0912260189652443,
 -0.03117534890770912,
 0.05269956961274147,
 0.04703487083315849,
 -0.08420306444168091,
 -0.030056191608309746,
 -0.020744830

In [32]:
print(len(vector))

384


In [33]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [34]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [35]:
os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY
os.environ["GROQ_API_KEY"]=GROQ_API_KEY

In [36]:
from pinecone import Pinecone
pinecone_api_key=PINECONE_API_KEY
pc=Pinecone(api_key=pinecone_api_key)

from pinecone import ServerlessSpec
index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,   # ✅ valid here
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index=pc.Index(index_name)
from langchain_pinecone import PineconeVectorStore
docsearch=PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name)

## I have already created pinecone index once so sot creating again,just connecting to the index and loading the required embeddings.

In [43]:

# Connect to your existing index
index = pc.Index("medical-chatbot")
# Verify connection (optional but recommended)
print(index.describe_index_stats())

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 7860}},
 'total_vector_count': 7860,
 'vector_type': 'dense'}


In [51]:
from langchain_pinecone import PineconeVectorStore

In [52]:
index_name="medical-chatbot"

In [44]:
# load existing documents from the index
docsearch=PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [45]:
docsearch

## Add more data to Pinecone Index
dswith=Document(
    page_content="This is a sample document for testing.",
    metadata={"source": "Youtube"}
)
docsearch.add_documents(documents=[dswith])

In [46]:
retriever=docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [47]:
retriever_docs=retriever.invoke("What is acne?")

In [48]:
retriever_docs

[Document(id='d9475c3e-a82c-44f8-8b20-b567e9175c7f', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='84f21d69-b37e-476b-96e2-bbfb2bc361ed', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='c22d9c7c-5e45-4e1b-880b-165eb092e571', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25')]

In [49]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # fast and free
    api_key=GROQ_API_KEY,
    temperature=0.5,
)

# Test
llm.invoke("What is acne?")

AIMessage(content="Acne is a common skin condition characterized by the occurrence of pimples, blackheads, whiteheads, and other inflammatory lesions on the skin, particularly on the face, chest, and back. It is caused by a combination of factors, including:\n\n1. **Overproduction of sebum**: The skin's oil glands produce an oily substance called sebum, which helps to keep the skin moisturized. However, when the glands produce too much sebum, it can clog pores and lead to acne.\n2. **Dead skin cells**: Dead skin cells can mix with sebum and clog pores, causing acne.\n3. **Bacteria**: A type of bacteria called Propionibacterium acnes (P. acnes) is naturally found on the skin and can contribute to the development of acne.\n4. **Hormonal changes**: Fluctuations in hormone levels, particularly during puberty, menstruation, pregnancy, and menopause, can lead to acne.\n5. **Genetics**: If your parents had acne, you may be more likely to develop it as well.\n6. **Stress**: Stress can increase

In [58]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [60]:


# ---- 1. Prompt ----
system_prompt = (
    "You are a Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)


In [ ]:
# ----  Format retrieved docs ----
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ---- RAG Chain ----
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)


In [62]:

# ---- Invoke ----
response = rag_chain.invoke("What are the symptoms of diabetes?")
print(response)

The symptoms of diabetes include fatigue and an abnormally high level of glucose in the blood (hyperglycemia). If left untreated, diabetes can damage or cause failure of the eyes, kidneys, nerves, heart, blood vessels, and other body organs. Common symptoms also include increased thirst and urination.


In [63]:
response = rag_chain.invoke("what is Acromegaly and gigantism?")
print(response)

Acromegaly is a disorder caused by the abnormal release of a chemical from the pituitary gland, leading to increased growth in bone and soft tissue, as well as various body disturbances. This condition can result in excessive growth, known as gigantism if it occurs before the end of puberty.


In [64]:
response = rag_chain.invoke("what is Acne?")
print(response)

Acne is a skin disorder in which the sebaceous glands become inflamed. It is characterized by the occurrence of pimples, spots, and other skin lesions. Acne affects both men and women, but is more common in adolescents and young adults.


In [65]:
response = rag_chain.invoke("what is the Treatment of Acne?")
print(response)

The treatment of acne depends on its severity and can include topical drugs such as tretinoin, benzoyl peroxide, adapalene, or salicylic acid to reduce the formation of new comedones. Topical antibiotics may be added to treat inflammation. Improvement is usually seen in two to four weeks.
